**String Functions**

| Function | What it does | Example |
|----------|--------------|---------|
| `upper(col)` | Converts all characters to uppercase | `"hello"` → `"HELLO"` |
| `lower(col)` | Converts all characters to lowercase | `"HELLO"` → `"hello"` |
| `initcap(col)` | Capitalizes the first letter of each word | `"hello world"` → `"Hello World"` |
| `trim(col)` | Removes leading and trailing spaces | `" hi "` → `"hi"` |
| `ltrim(col)` | Removes leading spaces only | `" hi "` → `"hi "` |
| `rtrim(col)` | Removes trailing spaces only | `" hi "` → `" hi"` |
| `length(col)` | Returns the number of characters | `"hello"` → `5` |
| `concat(c1, c2, ...)` | Joins multiple columns together | `"Hello"` + `" "` + `"World"` → `"Hello World"` |
| `concat_ws(sep, c1, c2, ...)` | Joins columns using a separator | `concat_ws(", ", city, state)` |
| `split(col, pattern)` | Splits a string into an array | `"a@b"` → `["a", "b"]` |
| `substring(col, pos, len)` | Extracts part of a string (`pos` starts at **1**) | `substring("O0001", 2, 4)` → `"0001"` |
| `regexp_replace(col, pattern, replacement)` | Replaces text using a regular expression | Remove digits, replace special characters, reformat codes |
| `col.contains(str)` | Checks if a string contains a substring | `col.contains("Card")` |
| `col.startswith(str)` | Checks if a string starts with a prefix | `col.startswith("O00")` |
| `col.endswith(str)` | Checks if a string ends with a suffix | `col.endswith("Card")` |

In [1]:
# Create Spark session
# Hadoop AWS connector allows Spark to communicate with Amazon S3
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Day-11")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)


:: loading settings :: url = jar:file:/opt/homebrew/Cellar/apache-spark/4.1.1/libexec/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rahulsinghrana/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rahulsinghrana/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ec1e04bd-9516-47e0-a169-4304b1fdd88e;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.780 in central
:: resolution report :: resolve 136ms :: artifacts dl 5ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.780 from central in [default]
	org.ap

**Task 1**

From customers.csv, create a full_name column by combining first_name and last_name using concat_ws(). Also create a location column combining city, state, and country separated by commas.

In [2]:
from pyspark.sql import functions as F

customers_df = spark.read.csv(
    "s3a://pyspark-30-days-rahul-2026/data/customers.csv",
    header=True,
    inferSchema=True
)

customers_df.withColumn(
    "full_name",
    F.concat_ws(
        " ",
        F.col("first_name"),
        F.col("last_name")
    )
).withColumn(
    "full_address",
    F.concat_ws(
        ", ",
        F.col("city"),
        F.col("state"),
        F.col("country")
    )
).show(truncate=False)


26/08/09 07:46:15 WARN CredentialProviderListFactory: Credentials option fs.s3a.aws.credentials.provider contains AWS v1 SDK entry com.amazonaws.auth.profile.ProfileCredentialsProvider; mapping to software.amazon.awssdk.auth.credentials.ProfileCredentialsProvider
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


+-----------+-----------+---------+-------------------------------+-------------+-----+-------+-----------+----------+---------------------+----------------------+
|customer_id|first_name |last_name|email                          |city         |state|country|signup_date|segment   |full_name            |full_address          |
+-----------+-----------+---------+-------------------------------+-------------+-----+-------+-----------+----------+---------------------+----------------------+
|C001       |James      |Anderson |james.anderson@email.com       |New York     |NY   |USA    |2021-03-15 |Enterprise|James Anderson       |New York, NY, USA     |
|C002       |Maria      |Garcia   |maria.garcia@email.com         |Los Angeles  |CA   |USA    |2021-05-22 |SMB       |Maria Garcia         |Los Angeles, CA, USA  |
|C003       |Robert     |Johnson  |robert.johnson@email.com       |Chicago      |IL   |USA    |2020-11-08 |Enterprise|Robert Johnson       |Chicago, IL, USA      |
|C004       |Lin


**Task 2**


From customers.csv, split the email column on @ to extract the username (before @) and domain (after @) as separate columns.

In [3]:
from pyspark.sql import functions as F

customers_df.withColumn(
    "username",
    F.split(F.col("email"), "@")[0]
).withColumn(
    "domain",
    F.split(F.col('email'),"@")[1]
).show(truncate=False)

+-----------+-----------+---------+-------------------------------+-------------+-----+-------+-----------+----------+---------------------+---------+
|customer_id|first_name |last_name|email                          |city         |state|country|signup_date|segment   |username             |domain   |
+-----------+-----------+---------+-------------------------------+-------------+-----+-------+-----------+----------+---------------------+---------+
|C001       |James      |Anderson |james.anderson@email.com       |New York     |NY   |USA    |2021-03-15 |Enterprise|james.anderson       |email.com|
|C002       |Maria      |Garcia   |maria.garcia@email.com         |Los Angeles  |CA   |USA    |2021-05-22 |SMB       |maria.garcia         |email.com|
|C003       |Robert     |Johnson  |robert.johnson@email.com       |Chicago      |IL   |USA    |2020-11-08 |Enterprise|robert.johnson       |email.com|
|C004       |Linda      |Martinez |linda.martinez@email.com       |Houston      |TX   |USA    

**Task 3**





From orders.csv, use regexp_replace() to reformat order_id — replace the leading O with ORD-. Show the original and reformatted values side by side.

In [4]:
from pyspark.sql import functions as F

orders_df = spark.read.csv(
    "s3a://pyspark-30-days-rahul-2026/data/orders.csv",
    header=True,
    inferSchema=True
)

orders_df.select(F.col('order_id'),
    F.regexp_replace(
        F.col("order_id"),
        "O",
        "ORD-"
    ).alias("New_order_id")
).show()


+--------+------------+
|order_id|New_order_id|
+--------+------------+
|   O0001|    ORD-0001|
|   O0002|    ORD-0002|
|   O0003|    ORD-0003|
|   O0004|    ORD-0004|
|   O0005|    ORD-0005|
|   O0006|    ORD-0006|
|   O0007|    ORD-0007|
|   O0008|    ORD-0008|
|   O0009|    ORD-0009|
|   O0010|    ORD-0010|
|   O0011|    ORD-0011|
|   O0012|    ORD-0012|
|   O0013|    ORD-0013|
|   O0014|    ORD-0014|
|   O0015|    ORD-0015|
|   O0016|    ORD-0016|
|   O0017|    ORD-0017|
|   O0018|    ORD-0018|
|   O0019|    ORD-0019|
|   O0020|    ORD-0020|
+--------+------------+
only showing top 20 rows


**Task 4**

From orders.csv, add a boolean column is_card_payment using contains("Card") on the payment_method column. How many orders were paid by card?

In [5]:
orders_df = orders_df.withColumn(
    "is_card_payment",
    F.col("payment_method").contains("Card")
)

orders_df.show()

+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+---------------+
|order_id|customer_id|product_id|order_date|quantity|unit_price|discount_pct|   status|payment_method| region|is_card_payment|
+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+---------------+
|   O0001|       C001|      P001|2023-01-05|       2|   1299.99|          10|Delivered|   Credit Card|   East|           true|
|   O0002|       C002|      P005|2023-01-07|       1|    449.99|           0|Delivered|        PayPal|   West|          false|
|   O0003|       C003|      P003|2023-01-10|       4|    349.99|          15|Delivered|   Credit Card|Midwest|           true|
|   O0004|       C004|      P006|2023-01-12|       2|     89.99|           5|Delivered|    Debit Card|  South|           true|
|   O0005|       C005|      P002|2023-01-15|       3|     29.99|           0|Delivered|   Credit Card|   West| 

In [6]:
spark.stop()